In [8]:
# node_shap_set.py
import numpy as np
import torch


def _get_device(model):
    return getattr(model, "device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


@torch.no_grad()
def _eval_gt_prob_single(
    model,
    x1,                  # torch [1,N,D]
    y1,                  # torch [1] (0/1)
    keep_mask_np,        # numpy [N] 1=present, 0=missing
    pad_mask1=None,      # torch [1,N] True=PAD, False=valid
    baseline="zeros",    # "zeros" or "bag_mean"
    positive_class_idx=1,
    clamp_eps=1e-6,
):
    """
    Evaluate P(GT) for a single bag under a coalition mask (present/absent per node).
    Missing nodes are replaced with baseline vector AND optionally masked out via pad_mask.
    """
    model.eval()
    device = _get_device(model)

    x1 = x1.to(device)
    y1 = y1.to(device).long().view(-1)  # [1]
    if pad_mask1 is not None:
        pad_mask1 = pad_mask1.to(device).bool()  # True=PAD

    _, N, D = x1.shape
    keep = torch.from_numpy(np.asarray(keep_mask_np, dtype=np.float32)).to(device)  # [N]
    keep3 = keep.view(1, N, 1)

    # baseline replacement
    if baseline == "zeros":
        base_vec = torch.zeros((1, 1, D), device=device, dtype=x1.dtype)
    elif baseline == "bag_mean":
        base_vec = x1.mean(dim=1, keepdim=True)
    else:
        raise ValueError("baseline must be 'zeros' or 'bag_mean'")

    x_m = x1 * keep3 + base_vec * (1.0 - keep3)

    # update pad mask: missing => PAD=True (since True means ignore)
    m_mask = None
    if pad_mask1 is not None:
        m_mask = pad_mask1.clone()
        missing = (keep < 0.5)
        m_mask[:, missing] = True

    logits = model(x_m, m_mask if pad_mask1 is not None else None)  # [1,2]
    p_pos = torch.softmax(logits, dim=1)[:, positive_class_idx]      # [1]
    p_true = torch.where(y1 == 1, p_pos, 1.0 - p_pos)                # [1]
    p = float(p_true.item())

    # harden numeric stability
    if not np.isfinite(p):
        p = 0.5
    p = min(max(p, clamp_eps), 1.0 - clamp_eps)
    return p


@torch.no_grad()
def node_shap_permutation_single(
    model,
    x1,                       # torch [1,N,D]
    y1,                       # torch [1]
    pad_mask1=None,           # torch [1,N] True=PAD, False=valid
    n_perms=128,
    baseline="zeros",         # "zeros" or "bag_mean"
    positive_class_idx=1,
    seed=0,
):
    """
    Robust Shapley approximation by random permutations (Monte Carlo).

    Returns
    -------
    phi_full : np.ndarray [N_full]
        Shapley value per node position (0 for padded nodes).
    base_value : float
        f(empty coalition) = P(GT) with no valid nodes present.
    valid_idx : np.ndarray
        indices of nodes that were valid (not padded) and explained.
    """
    model.eval()
    rng = np.random.default_rng(seed)

    device = _get_device(model)
    x1 = x1.to(device)
    y1 = y1.to(device).long().view(-1)

    # restrict to valid nodes only (recommended)
    if pad_mask1 is not None:
        pad_mask1 = pad_mask1.to(device).bool()  # True=PAD
        valid = (~pad_mask1).view(-1)            # True=valid
        valid_idx = torch.where(valid)[0].detach().cpu().numpy()
        if valid_idx.size == 0:
            N_full = x1.shape[1]
            return np.zeros((N_full,), np.float32), 0.5, valid_idx

        x_sub = x1[:, valid_idx, :]
        pm_sub = pad_mask1[:, valid_idx]  # should be all False, but keep
    else:
        valid_idx = np.arange(x1.shape[1])
        x_sub = x1
        pm_sub = None

    N = x_sub.shape[1]
    N_full = x1.shape[1]

    def f(mask01):
        return _eval_gt_prob_single(
            model=model,
            x1=x_sub,
            y1=y1,
            keep_mask_np=mask01,
            pad_mask1=pm_sub,
            baseline=baseline,
            positive_class_idx=positive_class_idx,
        )

    # base value: empty coalition (no nodes)
    empty = np.zeros((N,), dtype=np.float32)
    base_value = f(empty)

    phi = np.zeros((N,), dtype=np.float64)

    # Monte Carlo over permutations
    for _ in range(int(n_perms)):
        perm = rng.permutation(N)
        mask = np.zeros((N,), dtype=np.float32)
        prev = base_value
        for j in perm:
            mask[j] = 1.0
            cur = f(mask)
            phi[j] += (cur - prev)
            prev = cur

    phi /= float(n_perms)

    # expand back to full N
    phi_full = np.zeros((N_full,), dtype=np.float32)
    phi_full[valid_idx] = phi.astype(np.float32)

    return phi_full, float(base_value), valid_idx


@torch.no_grad()
def loo_node_importance_gt(
    model,
    batch,
    pad_mask_true_means_pad=True,  # keep True for your convention
    positive_class_idx=1,
    baseline="zeros",              # "zeros" or "bag_mean"
):
    """
    Your LOO (GT-prob drop) in a compact reusable form.

    Returns:
      delta_gt: [B,N]  p_true_base - p_true_i (positive => supports GT)
      p_true_base: [B]
      p_pos_base:  [B]
    """
    model.eval()
    device = _get_device(model)

    x = batch["features"].to(device)                 # [B,N,D]
    y = batch["labels"].to(device).long().view(-1)   # [B]
    pad_mask = batch.get("pad_mask", None)
    if pad_mask is not None:
        pad_mask = pad_mask.to(device).bool()        # True=PAD

    B, N, D = x.shape

    logits_base = model(x, pad_mask if pad_mask is not None else None)   # [B,2]
    p_pos_base = torch.softmax(logits_base, dim=1)[:, positive_class_idx]
    p_true_base = torch.where(y == 1, p_pos_base, 1.0 - p_pos_base)

    if baseline == "zeros":
        mean_vec = torch.zeros((B, 1, D), device=device, dtype=x.dtype)
    elif baseline == "bag_mean":
        mean_vec = x.mean(dim=1, keepdim=True)
    else:
        raise ValueError("baseline must be 'zeros' or 'bag_mean'")

    delta_gt = torch.zeros((B, N), device=device)

    for i in range(N):
        x_i = x.clone()
        x_i[:, i:i+1, :] = mean_vec

        m_i = None
        if pad_mask is not None:
            m_i = pad_mask.clone()
            if pad_mask_true_means_pad:
                m_i[:, i] = True
            else:
                m_i[:, i] = False

        logits_i = model(x_i, m_i if pad_mask is not None else None)
        p_pos_i = torch.softmax(logits_i, dim=1)[:, positive_class_idx]
        p_true_i = torch.where(y == 1, p_pos_i, 1.0 - p_pos_i)

        delta_gt[:, i] = (p_true_base - p_true_i)

    return (
        delta_gt.detach().cpu().numpy(),
        p_true_base.detach().cpu().numpy(),
        p_pos_base.detach().cpu().numpy(),
    )


In [9]:
import sys
sys.path.append("..")
from models.MIL import RadiomicsMIL
from dataloaders.deep_dataloaders import get_dataloaders_deep_learning
from utils import read_yaml_file, test_model, compute_classification_metrics, save_json, test_model_graph
import torch
import numpy as np
import os
from glob import glob
from pathlib import Path
import argparse 
from models.deep_sets import RadiomicsDeepSets
from models.transformer import RadiomicsTransformer
from models.MIL import RadiomicsMIL
from models.graph import RadiomicsGraph
from models.set_transformer import RadiomicsSetTransformer
from models.classical_ml import get_ml_models
from dataloaders.deep_dataloaders import get_dataloaders_deep_learning, get_center2_as_test_loader
from dataloaders.graph_dataloader import get_dataloaders_graph, get_center2_as_test_loader_graph
from dataloaders.ml_dataloaders import get_dataloaders_ml, get_classical_test_loader_center2
from utils import read_yaml_file, test_model, compute_classification_metrics, save_json, test_model_graph
from pathlib import Path

def model_generator(model_name: str):
    if model_name == "transformer":
        return RadiomicsTransformer
    elif model_name == "deep_sets":
        return RadiomicsDeepSets
    elif model_name == "mil":
        return RadiomicsMIL
    elif model_name == "graph":
        return RadiomicsGraph
    elif model_name == "set_transformer":
        return RadiomicsSetTransformer
    else:
        raise ValueError(f"Model {model_name} not found in model zoo.")

def data_function_generator_dl(args):
    if args.model_name in ["transformer", "deep_sets", "mil", "set_transformer"]:
        return get_dataloaders_deep_learning, get_center2_as_test_loader, test_model
    elif args.model_name == "graph":
        return get_dataloaders_graph, get_center2_as_test_loader_graph, test_model_graph
    else:
        raise ValueError(f"Model {args.model_name} not found in data function generator.")


def get_trained_model(args, fold_index: int):
    config_base_dir = '../configs'
    model_configs = read_yaml_file(Path(config_base_dir) / f"{args.model_name}.yaml")
    args.batch_size = model_configs['batch_size']
    print(args.use_coords, args.use_demographic)
    # get dataloaders
    get_data_loaders, get_center2_loader, test_model_func = data_function_generator_dl(args)
    train_loader, val_loader, test_loader = get_dataloaders_deep_learning(args, fold_index=fold_index)
    center2_loader = get_center2_loader(args)
    # define input dimension
    sample_batch = next(iter(train_loader))
    if args.model_name == "graph":
        input_dim = sample_batch.num_node_features
    else:
        input_dim = sample_batch['features'].shape[-1]
    print(f"Input dimension: {input_dim}")
    MODEL = model_generator(args.model_name)
    model = MODEL.load_from_checkpoint(args.best_model_path, 
                            input_dim=input_dim,
                            config=model_configs)   
    return model, test_loader    

class ARGS:
    def __init__(self, model_name: str, use_coords: bool, use_demographic: bool, fold_index: int):
        self.data_root = "/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/new_dataset"
        self.model_name = model_name
        self.use_coords = use_coords
        self.use_demographic = use_demographic
        self.batch_size = 32
        self.fold_index = fold_index
        model_postfic = "radiomics"
        if self.use_coords and not self.use_demographic:
            model_postfic = "coords_" + model_postfic
        elif self.use_demographic and not self.use_coords:
            model_postfic = "demographic_" + model_postfic
        elif self.use_coords and self.use_demographic:
            model_postfic = "coords_demographic_" + model_postfic
        
        self.best_model_path = f"/home/reza/Documents/Reza_projects/08_drarabi_lymphnodes/cleaned_code/Results/{self.model_name}_{model_postfic}/fold_{fold_index}/checkpoints/best.ckpt"





In [10]:
args = ARGS(model_name="mil", use_coords=True, use_demographic=True, fold_index=0)

model, test_loader = get_trained_model(args, args.fold_index)

True True
Train size: 102, Val size: 18, Test size: 31
Razavi Test size: 80
Input dimension: 112


In [14]:
from pathlib import Path
import numpy as np
import torch
from tqdm import tqdm



def extract_case_ids(batch):
    for k in ["patient_id", "pid", "case_id", "id"]:
        if k in batch:
            v = batch[k]
            if isinstance(v, (list, tuple)):
                return list(v)
            if torch.is_tensor(v):
                return [str(x) for x in v.detach().cpu().numpy().tolist()]
            return [str(x) for x in v]
    return None


@torch.no_grad()
def run_explanations_ragged(
    model,
    loader,
    out_path,
    max_cases=50,
    n_perms=128,
    baseline="zeros",
    also_run_loo=True,
    positive_class_idx=1,
    seed=0,
):
    model.eval()

    # store per-case, variable length
    phi_valid_list = []
    valid_idx_list = []
    N_full_list = []

    base_list = []
    y_list = []
    case_id_list = []

    loo_valid_list = []
    p_true_list = []
    p_pos_list = []

    case_count = 0

    for batch in tqdm(loader, desc="Explaining"):
        x = batch["features"]                 # [B,N,D] (N may vary across batches!)
        y = batch["labels"].long().view(-1)   # [B]
        pad_mask = batch.get("pad_mask", None)

        case_ids = extract_case_ids(batch)
        if case_ids is None:
            case_ids = [f"case_{case_count + i}" for i in range(x.shape[0])]

        B = x.shape[0]

        # compute batch LOO once if requested
        if also_run_loo:
            delta_gt, p_true_base, p_pos_base = loo_node_importance_gt(
                model,
                batch,
                pad_mask_true_means_pad=True,
                positive_class_idx=positive_class_idx,
                baseline=baseline,
            )

        for b in range(B):
            x1 = x[b:b+1]
            y1 = y[b:b+1]
            pm1 = pad_mask[b:b+1] if pad_mask is not None else None

            phi_full, base_value, valid_idx = node_shap_permutation_single(
                model=model,
                x1=x1,
                y1=y1,
                pad_mask1=pm1,
                n_perms=n_perms,
                baseline=baseline,
                positive_class_idx=positive_class_idx,
                seed=seed + case_count,
            )

            # save only valid entries to avoid padding/variable N issues
            valid_idx = np.asarray(valid_idx, dtype=np.int64)
            phi_valid = phi_full[valid_idx].astype(np.float32)

            phi_valid_list.append(phi_valid)
            valid_idx_list.append(valid_idx)
            N_full_list.append(int(phi_full.shape[0]))

            base_list.append(float(base_value))
            y_list.append(int(y1.item()))
            case_id_list.append(str(case_ids[b]))

            if also_run_loo:
                # delta_gt[b] is [N_full] for that batch; take only valid idx
                loo_valid_list.append(delta_gt[b][valid_idx].astype(np.float32))
                p_true_list.append(float(p_true_base[b]))
                p_pos_list.append(float(p_pos_base[b]))

            case_count += 1
            if case_count >= max_cases:
                break

        if case_count >= max_cases:
            break

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    save_dict = dict(
        phi_valid=np.asarray(phi_valid_list, dtype=object),
        valid_idx=np.asarray(valid_idx_list, dtype=object),
        N_full=np.asarray(N_full_list, dtype=np.int64),
        base_value=np.asarray(base_list, dtype=np.float32),
        y=np.asarray(y_list, dtype=np.int64),
        case_id=np.asarray(case_id_list, dtype=object),
        baseline=np.asarray([baseline], dtype=object),
        n_perms=np.asarray([n_perms], dtype=np.int64),
    )

    if also_run_loo:
        save_dict.update(dict(
            loo_valid=np.asarray(loo_valid_list, dtype=object),
            p_true_base=np.asarray(p_true_list, dtype=np.float32),
            p_pos_base=np.asarray(p_pos_list, dtype=np.float32),
        ))

    np.savez_compressed(str(out_path), **save_dict)
    print(f"Saved ragged explanations: {out_path} (cases={len(case_id_list)})")


In [34]:
model_name = "mil"   # "deep_sets" | "mil" | "transformer" | "set_transformer"
fold_index = 0
use_coords = True
use_demographic = True

# If your radiomics/features are standardized, use zeros baseline.
# If not standardized, try baseline="bag_mean" (more stable, closer to your LOO mean_vec idea).
baseline = "zeros"

max_cases = 50      # run on a subset first
n_perms = 128       # increase to 256/512 for more stable Shapley values

out_path = f"./explanations/{model_name}_fold{fold_index}_shap_perm.npz"

# --------- load model & loader from your code ----------
args = ARGS(model_name=model_name, use_coords=use_coords, use_demographic=use_demographic, fold_index=fold_index)
model, test_loader = get_trained_model(args, fold_index=fold_index)

out_path = f"./explanations/{model_name}_fold{fold_index}_shap_perm_ragged.npz"

run_explanations_ragged(
    model=model,
    loader=test_loader,
    out_path=out_path,
    max_cases=50,
    n_perms=128,
    baseline="zeros",
    also_run_loo=True,
    positive_class_idx=1,
    seed=0,
)


True True
Train size: 102, Val size: 18, Test size: 31
Razavi Test size: 80


Input dimension: 112


Explaining: 100%|██████████| 31/31 [00:32<00:00,  1.06s/it]

Saved ragged explanations: explanations/mil_fold0_shap_perm_ragged.npz (cases=31)


In [35]:
d = np.load(out_path, allow_pickle=True)
phi_valid = d["phi_valid"]
valid_idx = d["valid_idx"]
N_full = d["N_full"]

k = 0
phi_full = np.zeros((N_full[k],), dtype=np.float32)
phi_full[valid_idx[k]] = phi_valid[k]


In [36]:
import numpy as np

path = "./explanations/mil_fold0_shap_perm_ragged.npz"
d = np.load(path, allow_pickle=True)

phi_valid   = d["phi_valid"]     # list-like, len = #cases, each [Ni]
valid_idx   = d["valid_idx"]     # list-like, each [Ni]
N_full      = d["N_full"]        # [C]
base_value  = d["base_value"]    # [C]
y           = d["y"]             # [C]
case_id     = d["case_id"]       # [C]

# optional (if you saved LOO)
loo_valid   = d.get("loo_valid", None)
p_true_base = d.get("p_true_base", None)


In [37]:
def reconstruct_phi_full(k):
    phi = np.zeros((N_full[k],), dtype=np.float32)
    phi[valid_idx[k]] = phi_valid[k]
    return phi


In [48]:
k = 1
phi_full = reconstruct_phi_full(k)

print("Case:", case_id[k])
print("GT label:", y[k])
print("Base P(GT):", base_value[k])
print("Num nodes:", N_full[k])


Case: case_1
GT label: 1
Base P(GT): 0.5
Num nodes: 3


In [49]:
def top_k_nodes(phi, k=5):
    idx = np.argsort(np.abs(phi))[::-1]
    return idx, phi[idx]

idx, vals = top_k_nodes(phi_full)
for i, v in zip(idx, vals):
    print(f"Node {i:3d}: SHAP = {v:+.4f}")


Node   1: SHAP = +0.3123
Node   0: SHAP = -0.1851
Node   2: SHAP = -0.0469
